In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import pickle

from collections import Counter
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize
from scipy.sparse import hstack, csr_matrix
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df_preprocessed = pd.read_csv('downsampled.csv')

In [3]:
train_df, test_df = train_test_split(df_preprocessed, test_size=0.2, random_state=3, stratify=df_preprocessed['generated'])

In [4]:
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_train = vectorizer.fit_transform(train_df['text'])

In [5]:
def preprocess(df):
    def calculate_entropy(text_tokens):
        if not text_tokens:
            return 0.0
        counts = Counter(text_tokens)
        probabilities = [count / len(text_tokens) for count in counts.values()]
        entropy = -sum(p * math.log2(p) for p in probabilities)
        return entropy
    def calculate_burstiness(text):
        if not isinstance(text, str):
            return 0.0 # Handle non-string input
        sentences = sent_tokenize(text)
        if len(sentences) < 2:
            return 0.0 # Need at least 2 sentences to calculate variance
        sentence_lengths = [len(s.split()) for s in sentences]
        # Avoid division by zero if all sentence lengths are the same
        if len(set(sentence_lengths)) == 1:
            return 0.0
        return pd.Series(sentence_lengths).var()
    def calculate_lexical_diversity(text_tokens):
        if not text_tokens:
            return 0.0
        return len(set(text_tokens)) / len(text_tokens)
    df['entropy'] = df['tokenized_text'].apply(calculate_entropy)
    df['burstiness'] = df['text'].apply(calculate_burstiness)
    df['lexical_diversity'] = df['tokenized_text'].apply(calculate_lexical_diversity)

    tfidf = vectorizer.transform(df['text'])

    X_numeric = df[['entropy', 'burstiness', 'lexical_diversity']].values
    
    X = hstack([tfidf,csr_matrix(X_numeric)])
    return X

In [7]:
df_news

,text,year,tokenized_text,month_sin,month_cos,publisher
0,drugmakers from pfizer to gsk to hike us price...,2020,"['drugmakers', 'from', 'pfizer', 'to', 'gsk', ...",0.5,8.660254e-01,CNBC
1,iraqi militias start withdrawing from u.s. emb...,2020,"['iraqi', 'militias', 'start', 'withdrawing', ...",0.5,8.660254e-01,Reuters
2,update 1ghosn met lebanese president after fle...,2020,"['update', '1ghosn', 'met', 'lebanese', 'presi...",0.5,8.660254e-01,CNBC
3,heres what the world will look like in 2030 .....,2020,"['heres', 'what', 'the', 'world', 'will', 'loo...",0.5,8.660254e-01,Wired
4,5 tips to help you earn more cash while cleani...,2020,"['5', 'tips', 'to', 'help', 'you', 'earn', 'mo...",0.5,8.660254e-01,CNBC
...,...,...,...,...,...,...
248788,abm abm q2 earnings report preview what to loo...,2025,"['abm', 'abm', 'q2', 'earnings', 'report', 'pr...",-1.0,-1.836970e-16,StockStory
248789,eastern bank announces genuine foods as an ass...,2025,"['eastern', 'bank', 'announces', 'genuine', 'f...",-1.0,-1.836970e-16,Business Wire
248790,the top 5 analyst questions from dollar genera...,2025,"['the', 'top', '5', 'analyst', 'questions', 'f...",-1.0,-1.836970e-16,StockStory
248791,texpower appoints bader almonawer as president...,2025,"['texpower', 'appoints', 'bader', 'almonawer',...",-1.0,-1.836970e-16,GlobeNewswire


In [6]:
df_news = pd.read_csv('news_preprocessed_month.csv')

In [8]:
df_news.isna().sum()

text                  0
year                  0
tokenized_text        0
month_sin             0
month_cos             0
publisher         54915
dtype: int64

In [10]:
df_news.duplicated().sum()

np.int64(2523)

In [11]:
df_news=df_news.drop_duplicates()

In [12]:
df_news=df_news.dropna()

In [13]:
df_news_t = df_news[['text','tokenized_text']]

In [14]:
X_news = preprocess(df_news_t)

C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['entropy'] = df['tokenized_text'].apply(calculate_entropy)
C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['burstiness'] = df['text'].apply(calculate_burstiness)
C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

In [15]:
X_news

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 37429677 stored elements and shape (192801, 5003)>

In [16]:
tree = None  
with open('Tree fold 5.pkl', "rb") as f: # Take most recent
    tree = pickle.load(f)

In [18]:
predictions = tree.predict(X_news)

In [19]:
predictions

array([1., 1., 0., ..., 1., 1., 1.], shape=(192801,))

In [20]:
df_news['generated']=predictions

In [21]:
df_news.groupby(['generated']).count()

,text,year,tokenized_text,month_sin,month_cos,publisher
generated,,,,,,
0.0,56382,56382,56382,56382,56382,56382
1.0,136419,136419,136419,136419,136419,136419


In [22]:
df_news=df_news.drop(['text','tokenized_text'],axis='columns')

In [23]:
df_news

,year,month_sin,month_cos,publisher,generated
0,2020,0.5,8.660254e-01,CNBC,1.0
1,2020,0.5,8.660254e-01,Reuters,1.0
2,2020,0.5,8.660254e-01,CNBC,0.0
3,2020,0.5,8.660254e-01,Wired,0.0
4,2020,0.5,8.660254e-01,CNBC,0.0
...,...,...,...,...,...
248788,2025,-1.0,-1.836970e-16,StockStory,1.0
248789,2025,-1.0,-1.836970e-16,Business Wire,1.0
248790,2025,-1.0,-1.836970e-16,StockStory,1.0
248791,2025,-1.0,-1.836970e-16,GlobeNewswire,1.0


In [24]:
df_news.to_csv('df_news_generated.csv')

In [25]:
df_socmed = pd.read_csv('combined_socmed_month.csv')

In [26]:
df_socmed

,text,year,tokenized_text,month_sin,month_cos,source
0,this tiny little sterling item is actually a s...,2023,"['this', 'tiny', 'little', 'sterling', 'item',...",1.224647e-16,-1.000000e+00,youtube
1,music i music and you time music he music mus...,2020,"['music', 'i', 'music', 'and', 'you', 'time', ...",-8.660254e-01,5.000000e-01,youtube
2,redefine how to feel human redefine my kindnes...,2023,"['redefine', 'how', 'to', 'feel', 'human', 're...",5.000000e-01,-8.660254e-01,youtube
3,were here today in london to film with luke sm...,2024,"['were', 'here', 'today', 'in', 'london', 'to'...",1.000000e+00,6.123234e-17,youtube
4,hey chapel street church family im excited to ...,2023,"['hey', 'chapel', 'street', 'church', 'family'...",1.224647e-16,-1.000000e+00,youtube
...,...,...,...,...,...,...
96603,holy shit that pussy is so huge and perfect i ...,2025,"['holy', 'shit', 'that', 'pussy', 'is', 'so', ...",1.000000e+00,6.123234e-17,reddit
96604,that do didnt help him atall.\n\nmusta been so...,2025,"['that', 'do', 'didnt', 'help', 'him', 'atall....",1.000000e+00,6.123234e-17,reddit
96605,theyre not saying its not just news or that yo...,2025,"['theyre', 'not', 'saying', 'its', 'not', 'jus...",1.000000e+00,6.123234e-17,reddit
96606,lol awesome bro cheap shots at native carolini...,2025,"['lol', 'awesome', 'bro', 'cheap', 'shots', 'a...",1.000000e+00,6.123234e-17,reddit


In [27]:
df_socmed.isna().sum()

text              0
year              0
tokenized_text    0
month_sin         0
month_cos         0
source            0
dtype: int64

In [28]:
df_socmed.duplicated().sum()

np.int64(1517)

In [29]:
df_socmed=df_socmed.drop_duplicates()

In [30]:
df_socmed

,text,year,tokenized_text,month_sin,month_cos,source
0,this tiny little sterling item is actually a s...,2023,"['this', 'tiny', 'little', 'sterling', 'item',...",1.224647e-16,-1.000000e+00,youtube
1,music i music and you time music he music mus...,2020,"['music', 'i', 'music', 'and', 'you', 'time', ...",-8.660254e-01,5.000000e-01,youtube
2,redefine how to feel human redefine my kindnes...,2023,"['redefine', 'how', 'to', 'feel', 'human', 're...",5.000000e-01,-8.660254e-01,youtube
3,were here today in london to film with luke sm...,2024,"['were', 'here', 'today', 'in', 'london', 'to'...",1.000000e+00,6.123234e-17,youtube
4,hey chapel street church family im excited to ...,2023,"['hey', 'chapel', 'street', 'church', 'family'...",1.224647e-16,-1.000000e+00,youtube
...,...,...,...,...,...,...
96603,holy shit that pussy is so huge and perfect i ...,2025,"['holy', 'shit', 'that', 'pussy', 'is', 'so', ...",1.000000e+00,6.123234e-17,reddit
96604,that do didnt help him atall.\n\nmusta been so...,2025,"['that', 'do', 'didnt', 'help', 'him', 'atall....",1.000000e+00,6.123234e-17,reddit
96605,theyre not saying its not just news or that yo...,2025,"['theyre', 'not', 'saying', 'its', 'not', 'jus...",1.000000e+00,6.123234e-17,reddit
96606,lol awesome bro cheap shots at native carolini...,2025,"['lol', 'awesome', 'bro', 'cheap', 'shots', 'a...",1.000000e+00,6.123234e-17,reddit


In [31]:
df_socmed_t=df_socmed[['text','tokenized_text']]

In [32]:
X_socmed = preprocess(df_socmed_t)

C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['entropy'] = df['tokenized_text'].apply(calculate_entropy)
C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['burstiness'] = df['text'].apply(calculate_burstiness)
C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

In [33]:
X_socmed

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12347901 stored elements and shape (95091, 5003)>

In [34]:
predicted = tree.predict(X_socmed)

In [35]:
df_socmed['generated']=predicted

C:\Users\osten\AppData\Local\Temp\ipykernel_25788\298666550.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_socmed['generated']=predicted


In [36]:
df_socmed = df_socmed.drop(['tokenized_text','text'],axis='columns')

In [37]:
df_socmed

,year,month_sin,month_cos,source,generated
0,2023,1.224647e-16,-1.000000e+00,youtube,1.0
1,2020,-8.660254e-01,5.000000e-01,youtube,1.0
2,2023,5.000000e-01,-8.660254e-01,youtube,0.0
3,2024,1.000000e+00,6.123234e-17,youtube,1.0
4,2023,1.224647e-16,-1.000000e+00,youtube,0.0
...,...,...,...,...,...
96603,2025,1.000000e+00,6.123234e-17,reddit,1.0
96604,2025,1.000000e+00,6.123234e-17,reddit,1.0
96605,2025,1.000000e+00,6.123234e-17,reddit,0.0
96606,2025,1.000000e+00,6.123234e-17,reddit,1.0


In [38]:
df_socmed.pivot_table(
    index="source",
    columns="generated",
    values="year",            
    aggfunc="count",
    fill_value=0
).sort_values(by=[0, 1], ascending=False)

generated,0.0,1.0
source,,
youtube,23744,7361
reddit,4798,13226
twitter,791,44985
facebook,50,136


In [39]:
df_socmed.to_csv('socmed_generated_month.csv')

In [40]:
df_news

,year,month_sin,month_cos,publisher,generated
0,2020,0.5,8.660254e-01,CNBC,1.0
1,2020,0.5,8.660254e-01,Reuters,1.0
2,2020,0.5,8.660254e-01,CNBC,0.0
3,2020,0.5,8.660254e-01,Wired,0.0
4,2020,0.5,8.660254e-01,CNBC,0.0
...,...,...,...,...,...
248788,2025,-1.0,-1.836970e-16,StockStory,1.0
248789,2025,-1.0,-1.836970e-16,Business Wire,1.0
248790,2025,-1.0,-1.836970e-16,StockStory,1.0
248791,2025,-1.0,-1.836970e-16,GlobeNewswire,1.0


In [41]:
df_blogs = pd.read_csv('blogs_updated.csv')

In [42]:
df_blogs

,text,year,tokenized_text,month_sin,month_cos,source
0,this thanksgiving season feed your frankenstei...,2025,"['this', 'thanksgiving', 'season', 'feed', 'yo...",-0.866025,0.500000,Gizmodo
1,five nights at freddys 2 starring josh hutcher...,2025,"['five', 'nights', 'at', 'freddys', '2', 'star...",-0.500000,0.866025,Gizmodo
2,the company says its adding new guardrails to ...,2025,"['the', 'company', 'says', 'its', 'adding', 'n...",-0.866025,0.500000,Gizmodo
3,adam raines family members added troubling new...,2025,"['adam', 'raines', 'family', 'members', 'added...",-0.866025,0.500000,Gizmodo
4,its against facebooks terms of service to reve...,2025,"['its', 'against', 'facebooks', 'terms', 'of',...",-0.866025,0.500000,Gizmodo
...,...,...,...,...,...,...
30859,shocking truth the dark secrets behind western...,2023,"['shocking', 'truth', 'the', 'dark', 'secrets'...",-0.500000,0.866025,Medium
30860,catania food edition\n\nbeing in catania was t...,2022,"['catania', 'food', 'edition', 'being', 'in', ...",0.866025,0.500000,Medium
30861,we were almost something almost real but almos...,2024,"['we', 'were', 'almost', 'something', 'almost'...",-0.866025,-0.500000,Medium
30862,fog bathing\n\nan essay\n\nphoto by author joa...,2022,"['fog', 'bathing', 'an', 'essay', 'photo', 'by...",0.866025,-0.500000,Medium


In [44]:
df_blogs.duplicated().sum()

np.int64(541)

In [45]:
df_blogs=df_blogs.drop_duplicates()

In [46]:
df_blogs_t = df_blogs[['text','tokenized_text']]

In [47]:
X_blogs = preprocess(df_blogs_t)

C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['entropy'] = df['tokenized_text'].apply(calculate_entropy)
C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['burstiness'] = df['text'].apply(calculate_burstiness)
C:\Users\osten\AppData\Local\Temp\ipykernel_25788\4263096284.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

In [48]:
predicted = tree.predict(X_blogs)

In [49]:
predicted

array([1., 0., 0., ..., 0., 1., 1.], shape=(30323,))

In [50]:
df_blogs

,text,year,tokenized_text,month_sin,month_cos,source
0,this thanksgiving season feed your frankenstei...,2025,"['this', 'thanksgiving', 'season', 'feed', 'yo...",-0.866025,5.000000e-01,Gizmodo
1,five nights at freddys 2 starring josh hutcher...,2025,"['five', 'nights', 'at', 'freddys', '2', 'star...",-0.500000,8.660254e-01,Gizmodo
2,the company says its adding new guardrails to ...,2025,"['the', 'company', 'says', 'its', 'adding', 'n...",-0.866025,5.000000e-01,Gizmodo
3,adam raines family members added troubling new...,2025,"['adam', 'raines', 'family', 'members', 'added...",-0.866025,5.000000e-01,Gizmodo
4,its against facebooks terms of service to reve...,2025,"['its', 'against', 'facebooks', 'terms', 'of',...",-0.866025,5.000000e-01,Gizmodo
...,...,...,...,...,...,...
30858,the clean good life a terminal diagnosis and ...,2020,"['the', 'clean', 'good', 'life', 'a', 'termina...",1.000000,6.123234e-17,Medium
30860,catania food edition\n\nbeing in catania was t...,2022,"['catania', 'food', 'edition', 'being', 'in', ...",0.866025,5.000000e-01,Medium
30861,we were almost something almost real but almos...,2024,"['we', 'were', 'almost', 'something', 'almost'...",-0.866025,-5.000000e-01,Medium
30862,fog bathing\n\nan essay\n\nphoto by author joa...,2022,"['fog', 'bathing', 'an', 'essay', 'photo', 'by...",0.866025,-5.000000e-01,Medium


In [51]:
df_blogs['generated']=predicted

C:\Users\osten\AppData\Local\Temp\ipykernel_25788\3382044250.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_blogs['generated']=predicted


In [52]:
df_blogs

,text,year,tokenized_text,month_sin,month_cos,source,generated
0,this thanksgiving season feed your frankenstei...,2025,"['this', 'thanksgiving', 'season', 'feed', 'yo...",-0.866025,5.000000e-01,Gizmodo,1.0
1,five nights at freddys 2 starring josh hutcher...,2025,"['five', 'nights', 'at', 'freddys', '2', 'star...",-0.500000,8.660254e-01,Gizmodo,0.0
2,the company says its adding new guardrails to ...,2025,"['the', 'company', 'says', 'its', 'adding', 'n...",-0.866025,5.000000e-01,Gizmodo,0.0
3,adam raines family members added troubling new...,2025,"['adam', 'raines', 'family', 'members', 'added...",-0.866025,5.000000e-01,Gizmodo,0.0
4,its against facebooks terms of service to reve...,2025,"['its', 'against', 'facebooks', 'terms', 'of',...",-0.866025,5.000000e-01,Gizmodo,0.0
...,...,...,...,...,...,...,...
30858,the clean good life a terminal diagnosis and ...,2020,"['the', 'clean', 'good', 'life', 'a', 'termina...",1.000000,6.123234e-17,Medium,0.0
30860,catania food edition\n\nbeing in catania was t...,2022,"['catania', 'food', 'edition', 'being', 'in', ...",0.866025,5.000000e-01,Medium,0.0
30861,we were almost something almost real but almos...,2024,"['we', 'were', 'almost', 'something', 'almost'...",-0.866025,-5.000000e-01,Medium,0.0
30862,fog bathing\n\nan essay\n\nphoto by author joa...,2022,"['fog', 'bathing', 'an', 'essay', 'photo', 'by...",0.866025,-5.000000e-01,Medium,1.0


In [53]:
df_blogs = df_blogs.drop(['text','tokenized_text'],axis='columns')

In [54]:
df_blogs.to_csv('blogs_generated_month.csv')

In [55]:
df_socmed

,year,month_sin,month_cos,source,generated
0,2023,1.224647e-16,-1.000000e+00,youtube,1.0
1,2020,-8.660254e-01,5.000000e-01,youtube,1.0
2,2023,5.000000e-01,-8.660254e-01,youtube,0.0
3,2024,1.000000e+00,6.123234e-17,youtube,1.0
4,2023,1.224647e-16,-1.000000e+00,youtube,0.0
...,...,...,...,...,...
96603,2025,1.000000e+00,6.123234e-17,reddit,1.0
96604,2025,1.000000e+00,6.123234e-17,reddit,1.0
96605,2025,1.000000e+00,6.123234e-17,reddit,0.0
96606,2025,1.000000e+00,6.123234e-17,reddit,1.0


In [56]:
df_blogs

,year,month_sin,month_cos,source,generated
0,2025,-0.866025,5.000000e-01,Gizmodo,1.0
1,2025,-0.500000,8.660254e-01,Gizmodo,0.0
2,2025,-0.866025,5.000000e-01,Gizmodo,0.0
3,2025,-0.866025,5.000000e-01,Gizmodo,0.0
4,2025,-0.866025,5.000000e-01,Gizmodo,0.0
...,...,...,...,...,...
30858,2020,1.000000,6.123234e-17,Medium,0.0
30860,2022,0.866025,5.000000e-01,Medium,0.0
30861,2024,-0.866025,-5.000000e-01,Medium,0.0
30862,2022,0.866025,-5.000000e-01,Medium,1.0


In [57]:
df_news

,year,month_sin,month_cos,publisher,generated
0,2020,0.5,8.660254e-01,CNBC,1.0
1,2020,0.5,8.660254e-01,Reuters,1.0
2,2020,0.5,8.660254e-01,CNBC,0.0
3,2020,0.5,8.660254e-01,Wired,0.0
4,2020,0.5,8.660254e-01,CNBC,0.0
...,...,...,...,...,...
248788,2025,-1.0,-1.836970e-16,StockStory,1.0
248789,2025,-1.0,-1.836970e-16,Business Wire,1.0
248790,2025,-1.0,-1.836970e-16,StockStory,1.0
248791,2025,-1.0,-1.836970e-16,GlobeNewswire,1.0


In [58]:
result_socmed = df_socmed.groupby('source')['generated'].agg([
    ('0_count', lambda x: (x == 0).sum()),
    ('1_count', lambda x: (x == 1).sum()),
    ('pct_of_1', lambda x: (x == 1).mean() * 100)
]).reset_index()

result_socmed.columns = ['source', '0_count', '1_count', '% of 1/total']
result_socmed

,source,0_count,1_count,% of 1/total
0,facebook,50,136,73.118280
1,reddit,4798,13226,73.379938
2,twitter,791,44985,98.272020
3,youtube,23744,7361,23.665006


In [59]:
result_blogs = df_blogs.groupby('source')['generated'].agg([
    ('0_count', lambda x: (x == 0).sum()),
    ('1_count', lambda x: (x == 1).sum()),
    ('pct_of_1', lambda x: (x == 1).mean() * 100)
]).reset_index()

result_blogs.columns = ['source', '0_count', '1_count', '% of 1/total']
result_blogs

,source,0_count,1_count,% of 1/total
0,Gizmodo,8738,3341,27.659574
1,MLMastery,923,371,28.670788
2,Medium,3333,1701,33.790226
3,lifehacker,10558,1358,11.396442


In [60]:
result_news = df_news.groupby('publisher')['generated'].agg([
    ('0_count', lambda x: (x == 0).sum()),
    ('1_count', lambda x: (x == 1).sum()),
    ('pct_of_1', lambda x: (x == 1).mean() * 100)
]).reset_index()

result_news.columns = ['source', '0_count', '1_count', '% of 1/total']
result_news.sort_values(by='1_count')

,source,0_count,1_count,% of 1/total
2773,['DNN'],1,0,0.000000
2762,['DLPN'],1,0,0.000000
2772,['DMTK'],4,0,0.000000
2764,['DLTH'],3,0,0.000000
2470,['CHY'],1,0,0.000000
...,...,...,...,...
1699,['ACGLO'],1231,3395,73.389537
1486,GlobeNewswire,38,3867,99.026889
1592,Reuters,1067,3875,78.409551
1659,Zacks,412,4145,90.958964


In [70]:
def aggregate(df):
    aggregated = df.groupby(['year', 'month_sin', 'month_cos']).agg({
        'generated': ['sum', 'count']
    }).reset_index()
    
    aggregated.columns = ['year', 'month_sin', 'month_cos', 'ai_count', 'total_count']
    aggregated['ai_percentage'] = (aggregated['ai_count'] / aggregated['total_count']) * 100
    
    aggregated = aggregated.sort_values(['year']).reset_index(drop=True)

    aggregated['ai_pct_lag1'] = aggregated['ai_percentage'].shift(1)
    aggregated['ai_pct_lag2'] = aggregated['ai_percentage'].shift(2)
    aggregated['ai_pct_ma3'] = aggregated['ai_percentage'].rolling(window=3).mean()

    aggregated = aggregated.drop(['ai_count','total_count'],axis='columns')
    aggregated["time_index"] = np.arange(len(aggregated))
    return aggregated

In [71]:
aggregated_news = aggregate(df_news)
aggregated_socmed = aggregate(df_socmed)
aggregated_blog = aggregate(df_blogs)

In [72]:
aggregated_news

,year,month_sin,month_cos,ai_percentage,ai_pct_lag1,ai_pct_lag2,ai_pct_ma3,time_index
0,2020,-1.000000e+00,-1.836970e-16,66.632860,NaN,NaN,NaN,0
1,2020,-5.000000e-01,8.660254e-01,69.531250,66.632860,NaN,NaN,1
2,2020,-5.000000e-01,-8.660254e-01,74.539877,69.531250,66.632860,70.234662,2
3,2020,-2.449294e-16,1.000000e+00,60.782025,74.539877,69.531250,68.284384,3
4,2020,1.224647e-16,-1.000000e+00,98.857143,60.782025,74.539877,78.059682,4
...,...,...,...,...,...,...,...,...
58,2025,5.000000e-01,-8.660254e-01,83.790698,83.054004,88.262599,85.035767,58
59,2025,5.000000e-01,8.660254e-01,60.916767,83.790698,83.054004,75.920490,59
60,2025,8.660254e-01,5.000000e-01,62.055838,60.916767,83.790698,68.921101,60
61,2025,8.660254e-01,-5.000000e-01,80.631244,62.055838,60.916767,67.867950,61


In [73]:
aggregated_socmed

,year,month_sin,month_cos,ai_percentage,ai_pct_lag1,ai_pct_lag2,ai_pct_ma3,time_index
0,2020,-1.000000,-1.836970e-16,86.458333,NaN,NaN,NaN,0
1,2020,-1.000000,-1.836970e-16,61.494720,86.458333,NaN,NaN,1
2,2020,-0.866025,5.000000e-01,69.779507,61.494720,86.458333,72.577520,2
3,2020,-0.866025,-5.000000e-01,64.179104,69.779507,61.494720,65.151110,3
4,2020,-0.500000,8.660254e-01,71.245283,64.179104,69.779507,68.401298,4
...,...,...,...,...,...,...,...,...
65,2025,0.500000,-8.660254e-01,71.232877,55.054903,31.370239,52.552673,65
66,2025,0.500000,8.660254e-01,62.625074,71.232877,55.054903,62.970951,66
67,2025,0.866025,5.000000e-01,68.726054,62.625074,71.232877,67.528001,67
68,2025,0.866025,-5.000000e-01,73.550296,68.726054,62.625074,68.300474,68


In [74]:
aggregated_blog

,year,month_sin,month_cos,ai_percentage,ai_pct_lag1,ai_pct_lag2,ai_pct_ma3,time_index
0,2020,-1.000000,-1.836970e-16,22.222222,NaN,NaN,NaN,0
1,2020,-1.000000,-1.836970e-16,15.539568,22.222222,NaN,NaN,1
2,2020,-0.866025,5.000000e-01,10.869565,15.539568,22.222222,16.210452,2
3,2020,-0.866025,-5.000000e-01,11.132438,10.869565,15.539568,12.513857,3
4,2020,-0.500000,8.660254e-01,12.648221,11.132438,10.869565,11.550075,4
...,...,...,...,...,...,...,...,...
71,2025,0.500000,-8.660254e-01,25.925926,27.040816,29.922780,27.629841,71
72,2025,0.500000,8.660254e-01,21.942446,25.925926,27.040816,24.969729,72
73,2025,0.866025,5.000000e-01,23.303167,21.942446,25.925926,23.723846,73
74,2025,0.866025,-5.000000e-01,33.895131,23.303167,21.942446,26.380248,74


In [75]:
aggregated_blog=aggregated_blog.dropna()
aggregated_news=aggregated_news.dropna()
aggregated_socmed=aggregated_socmed.dropna()

In [76]:
aggregated_news.to_csv('aggregate_news.csv',index=False)

In [77]:
aggregated_blog.to_csv('aggregate_blog.csv',index=False)

In [78]:
aggregated_socmed.to_csv('aggregated_socmed.csv',index=False)